# 02_clean_prices.py 결과 확인

`data/cleaned/prices`의 결측 보간(`is_interpolated`) / 액면분할 의심 탐지(`split_suspected`) 결과를 확인한다.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.master("spark://spark-master:7077").appName("check_cleaned_prices").getOrCreate()

df = spark.read.parquet("/opt/spark-apps/data/cleaned/prices")
print(f"전체 {df.count()}건")
df.groupBy("snapshot_type").count().toPandas()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/04 08:22:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


전체 1797973건


,snapshot_type,count
0,current,1774650
1,12m_ago,11477
2,1m_ago,11846


## 1. 보간된 행 (`is_interpolated=true`)
직전 종가로 채운 결측 거래일. 거래정지 구간도 원본에 시세가 없는 형태로 들어오므로 여기 포함된다.

In [2]:
interpolated = df.filter(F.col("is_interpolated"))
print(f"보간된 행: {interpolated.count()}건")
interpolated.select("stock_code", "bas_dt", "close_price", "is_interpolated") \
    .orderBy("stock_code", "bas_dt") \
    .toPandas()

보간된 행: 0건


,stock_code,bas_dt,close_price,is_interpolated


## 2. 액면분할/병합 의심 행 (`split_suspected=true`)
전일 대비 종가 등락률이 -40% 이하 또는 +67% 이상인 지점. 자동 보정 없이 플래그만 표시됨 — 진짜 분할/병합인지, 단순 급등락인지는 별도 확인 필요.

In [3]:
w = Window.partitionBy("stock_code").orderBy("bas_dt")
suspects = df.withColumn("prev_close", F.lag("close_price").over(w)) \
    .filter(F.col("split_suspected")) \
    .withColumn("day_over_day_rate", F.round((F.col("close_price") - F.col("prev_close")) / F.col("prev_close") * 100, 2))

print(f"액면분할/병합 의심 행: {suspects.count()}건")
suspects.select("stock_code", "bas_dt", "prev_close", "close_price", "day_over_day_rate") \
    .orderBy("stock_code", "bas_dt") \
    .toPandas()

액면분할/병합 의심 행: 1960건


,stock_code,bas_dt,prev_close,close_price,day_over_day_rate
0,000020,20260710,5050,5160,2.18
1,000040,20200224,242,656,171.07
2,000040,20260728,267,1047,292.13
3,000100,20200424,229000,46950,-79.50
4,000120,20260710,77100,73900,-4.15
...,...,...,...,...,...
1955,950130,20260710,2305,2500,8.46
1956,950160,20221025,8010,20850,160.30
1957,950160,20260710,97400,90000,-7.60
1958,950200,20201123,11600,21150,82.33


## 3. 정제 후에도 남은 결측치 (컬럼별 null 개수)
전부 0이어야 정상.

In [4]:
null_counts = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns])
null_counts.toPandas()

,stock_code,bas_dt,close_price,fluctuation_rate,listed_share_count,market_cap,open_price,snapshot_type,is_interpolated,split_suspected,year
0,0,0,0,0,0,0,0,0,0,0,0


In [5]:
spark.stop()